# Anveshan NLP — train the deployable MuRIL intent model
Trains **one** `google/muril-base-cased` on **all** training rows (romanized + chat augmentation + pseudo-labels)
and saves it as `muril_intent/` (+ `muril_intent.zip`) for the FastAPI demo in `deploy/`.
GPU: Kaggle T4 (single GPU is enough) or Colab T4. ~40 min. **Does not submit anything.**

In [ ]:
# ---- CONFIG ----
MODEL, LR, EPOCHS, BS, MAX_LEN, SEED = 'google/muril-base-cased', 2e-5, 5, 32, 64, 42
PSEUDO_FILE = 'pseudo_labels.csv'   # set None to skip pseudo-labels

In [ ]:
import os, glob
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if ON_KAGGLE:
    OUT = '/kaggle/working'
else:
    from google.colab import drive; drive.mount('/content/drive'); OUT = '/content/drive/MyDrive/anveshan_deploy'
os.makedirs(OUT, exist_ok=True)
def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    return hits[0] if hits else name
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| output dir:', OUT)

In [ ]:
%%writefile common.py
import re
V = {'अ':'a','आ':'aa','इ':'i','ई':'ee','उ':'u','ऊ':'oo','ऋ':'ri','ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऑ':'o','ऍ':'e'}
M = {'ा':'aa','ि':'i','ी':'ee','ु':'u','ू':'oo','ृ':'ri','े':'e','ै':'ai','ो':'o','ौ':'au','ॉ':'o','ॅ':'e'}
C = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'n','च':'ch','छ':'chh','ज':'j','झ':'jh','ञ':'n','ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n',
     'त':'t','थ':'th','द':'d','ध':'dh','न':'n','प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v','श':'sh','ष':'sh',
     'स':'s','ह':'h','क़':'q','ख़':'kh','ग़':'g','ज़':'z','ड़':'d','ढ़':'rh','फ़':'f','ळ':'l'}
NUK = {'क':'q','ख':'kh','ग':'g','ज':'z','ड':'d','ढ':'rh','फ':'f'}
def _word(w, drop_schwa=True):
    out = []; i = 0; n = len(w)
    while i < n:
        ch = w[i]
        if ch in C:
            base = C[ch]
            if i+1 < n and w[i+1] == '़': base = NUK.get(ch, base); i += 1
            nxt = w[i+1] if i+1 < n else ''
            if nxt == '्': out.append(base); i += 2; continue
            if nxt in M: out.append(base + M[nxt]); i += 2; continue
            out.append(base + 'a'); i += 1; continue
        if ch in V: out.append(V[ch])
        elif ch in 'ंँ': out.append('n')
        elif ch == 'ः': out.append('h')
        elif '०' <= ch <= '९': out.append(str(ord(ch) - ord('०')))
        elif ch in '़्।': pass
        else: out.append(ch)
        i += 1
    s = ''.join(out)
    if drop_schwa and len(s) > 3 and s.endswith('a') and not s.endswith('aa') and w[-1] in C:
        s = s[:-1]
    return s
def romanize(text, drop_schwa=True):
    return ' '.join(_word(w, drop_schwa) for w in text.split()).lower()

import pandas as pd
def light(s):  # casual short-vowel spelling: 'kripayaa' -> 'kripaya', 'eemel' -> 'imel'
    return s.replace('aa', 'a').replace('ee', 'i').replace('oo', 'u')
def is_dev(s):
    return bool(re.search('[ऀ-ॿ]', s))
def norm(s):
    s = str(s).strip()
    return s if is_dev(s) else s.lower()
import random as _random
_SPELL = [(r'\bkya\b', ['kia', 'kya', 'kyaa']), (r'\bhai\b', ['h', 'hai', 'he']), (r'\bhain\b', ['h', 'hain', 'hai']),
          (r'\bmujhe\b', ['mjhe', 'mujhe', 'muje']), (r'\bnahin?\b', ['nhi', 'nahi', 'nai']), (r'\bkaro\b', ['kro', 'karo', 'kar do']),
          (r'\bkaren\b', ['kare', 'karo', 'kren']), (r'\bkarein\b', ['kare', 'karo']), (r'\baap\b', ['ap', 'aap', 'tum']),
          (r'\bmein\b', ['me', 'mai', 'mein']), (r'\bmen\b', ['me', 'mein']), (r'\bkripaya\b', ['please', 'plz', 'kripya']),
          (r'\bkripayaa\b', ['please', 'plz']), (r'\bke liye\b', ['ke liye', 'ke lie', 'k liye']), (r'\bbhi\b', ['bhi', 'b']),
          (r'\bthik\b', ['theek', 'thik']), (r'\baur\b', ['or', 'aur'])]
_EN = [(r'\bbatti(?:yaan|yan|yon|yaa)?\b', 'lights'), (r'\broshanee\b|\broshni\b', 'light'), (r'\bchaal?oo?\b', 'on'),
       (r'\bband\b', 'off'), (r'\bsangeet\b', 'music'), (r'\bgaanaa?\b', 'song'), (r'\bgaane\b', 'songs'),
       (r'\bmausam\b', 'weather'), (r'\bsamay\b', 'time'), (r'\bsandesh\b', 'message'), (r'\bbhejo\b', 'send karo'),
       (r'\bhataa?o\b', 'delete karo'), (r'\bhataa?en\b', 'delete karo'), (r'\byaad dilaa?o\b', 'remind karo'),
       (r'\bsamaachaar\b|\bkhabar\b', 'news'), (r'\bkal\b', 'tomorrow'), (r'\baaj\b', 'today'), (r'\bghar\b', 'home'),
       (r'\btaareekh\b|\btarikh\b', 'date'), (r'\bsoochee\b|\bsuchi\b', 'list'), (r'\bphon\b', 'phone'),
       (r'\bvolyoom\b|\bvolyum\b', 'volume'), (r'\bkailendar\b', 'calendar'), (r'\beemel\b|\bimel\b', 'email')]
_FILL_PRE = ['', '', '', 'please ', 'plz ', 'google ', 'alexa ', 'bhai ', 'yaar ', 'hey ']
_FILL_SUF = ['', '', '', ' please', ' bhai', ' na', ' yaar', ' plz', ' jaldi', ' abhi']
def chatify(s, rng):
    """Casual Hinglish rewrite of a romanized query: texting spellings, English swaps, fillers."""
    for pat, opts in _SPELL:
        if rng.random() < 0.6: s = re.sub(pat, lambda m: rng.choice(opts), s)
    for pat, rep in _EN:
        if rng.random() < 0.5: s = re.sub(pat, rep, s)
    if rng.random() < 0.3: s = light(s)
    s = rng.choice(_FILL_PRE) + s + rng.choice(_FILL_SUF)
    return re.sub(r'\s+', ' ', s).strip()
def augment(df, seed=42, chat=True, pseudo=None):
    """Training set = originals + romanized (full/light) copies of Devanagari rows
    + one casual 'chat' copy of every romanized row + optional pseudo-labelled test rows."""
    rows = [df[['query_text', 'y']]]
    d = df[~df.roman]
    r1 = d.query_text.map(romanize); r2 = r1.map(light)
    rows += [pd.DataFrame({'query_text': r1, 'y': d.y}), pd.DataFrame({'query_text': r2, 'y': d.y})]
    if pseudo is not None and len(pseudo):
        rows.append(pseudo[['query_text', 'y']])
    out = pd.concat(rows)
    if chat:
        rng = _random.Random(seed)
        ro = out[~out.query_text.map(is_dev)]
        rows.append(pd.DataFrame({'query_text': [chatify(t, rng) for t in ro.query_text], 'y': ro.y.values}))
        out = pd.concat([out, rows[-1]])
    out = out.drop_duplicates(['query_text', 'y']).sample(frac=1, random_state=seed)
    return out.reset_index(drop=True)

import collections
def _key(s):
    s = light(s.lower()); s = re.sub(r'[^a-z0-9 ]', '', s); s = re.sub(r'(.)\1+', r'\1', s)
    return s.replace('w', 'v').replace('sh', 's').replace('ph', 'f').replace('z', 'j')
def _grams(s):
    s = ' ' + s + ' '; return {s[i:i+3] for i in range(len(s) - 2)}
def twin_groups(texts, thresh=0.5):
    """Group id per row: a romanized row shares its group with its most similar Devanagari row
    (its transliteration twin), so CV folds never split a sentence from its twin."""
    texts = list(texts); groups = list(range(len(texts)))
    dv = [i for i, t in enumerate(texts) if is_dev(t)]
    G = {i: _grams(_key(romanize(texts[i]))) for i in dv}
    idx = collections.defaultdict(list)
    for i in dv:
        for t in G[i]: idx[t].append(i)
    for j, t in enumerate(texts):
        if is_dev(t): continue
        g = _grams(_key(t)); cnt = collections.Counter()
        for x in g:
            for i in idx.get(x, ()): cnt[i] += 1
        if not cnt: continue
        s, i = max((c / (len(g) + len(G[i]) - c), i) for i, c in cnt.items())
        if s >= thresh: groups[j] = groups[i]
    return groups


In [ ]:
%%writefile labels.json
{
 "note": "Intent names inferred from training examples (the competition only gives numeric ids 0-55). Id 55 is a mixed bucket in the data: alarm set/query, podcasts, coffee and games.",
 "names": [
  "datetime_query", "light_change_colour", "train_ticket_booking", "food_order_status", "stock_price",
  "greeting", "local_events", "music_dislike", "smart_plug_off", "cooking_recipe",
  "currency_exchange", "traffic_query", "general_chitchat", "weather_query", "volume_up",
  "email_add_contact", "food_order", "contact_lookup", "light_brighten", "place_recommendation",
  "audiobook_play", "list_create_or_add", "news_query", "smart_plug_on", "joke",
  "definition_query", "social_media_query", "music_settings", "audio_settings", "calendar_remove",
  "light_dim", "calendar_query", "email_send", "vacuum_cleaner", "volume_down",
  "radio_play", "cooking_tips", "timezone_convert", "maths", "light_off",
  "light_on", "transport_route_query", "music_like", "email_check", "music_play",
  "volume_mute", "social_media_post", "general_knowledge", "calendar_reminder_set", "alarm_remove",
  "list_remove", "taxi_booking", "movie_recommendation", "song_info", "list_query",
  "alarm_podcast_coffee_game"
 ]
}


In [ ]:
import json, random, time, shutil, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from common import augment, is_dev
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
train = pd.read_csv(find('train.csv'), keep_default_na=False)
classes = np.sort(train.target.unique()); c2i = {c: i for i, c in enumerate(classes)}
train['y'] = train.target.map(c2i); train['roman'] = ~train.query_text.map(is_dev); NC = len(classes)
pseudo = None
if PSEUDO_FILE and os.path.exists(find(PSEUDO_FILE)):
    pseudo = pd.read_csv(find(PSEUDO_FILE), keep_default_na=False)[['query_text', 'y']]
    print('pseudo rows:', len(pseudo))
tr = augment(train, SEED, pseudo=pseudo)
print('train rows:', len(train), '-> augmented:', len(tr), '| classes:', NC)
DEV = 'cuda'
tok = AutoTokenizer.from_pretrained(MODEL)
id2label = {i: n for i, n in enumerate(json.load(open('labels.json'))['names'])}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=NC, id2label=id2label, label2id={v: k for k, v in id2label.items()}).to(DEV)
def col(b):
    enc = tok([x[0] for x in b], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    enc['labels'] = torch.tensor([x[1] for x in b]); return enc
dl = torch.utils.data.DataLoader(list(zip(tr.query_text, tr.y)), batch_size=BS, shuffle=True, collate_fn=col, num_workers=1)
nd = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
opt = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if not any(x in n for x in nd)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(x in n for x in nd)], 'weight_decay': 0.0}], lr=LR)
steps = len(dl) * EPOCHS; sch = get_cosine_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler('cuda'); lossf = nn.CrossEntropyLoss(label_smoothing=0.1); t0 = time.time()
for ep in range(EPOCHS):
    model.train(); run = 0
    for i, b in enumerate(dl):
        b = {k: v.to(DEV) for k, v in b.items()}; y = b.pop('labels')
        with torch.autocast('cuda', dtype=torch.float16):
            loss = lossf(model(**b).logits.float(), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); run += loss.item()
    print(f'epoch {ep} loss {run / len(dl):.4f} ({time.time() - t0:.0f}s)', flush=True)

In [ ]:
# sanity check on a few training sentences, then save (fp16 weights: ~1 GB instead of ~2 GB)
model.eval()
for s in ['kal subah 7 baje alarm lagao', 'aaj mausam kaisa hai', 'arijit singh ke gaane chalao', 'मेरे नए ईमेल दिखाओ']:
    with torch.no_grad():
        p = torch.softmax(model(**tok(s, return_tensors='pt').to(DEV)).logits.float(), -1)[0]
    print(f'{s!r:40} -> {id2label[int(p.argmax())]} ({float(p.max()):.2f})')
save_dir = f'{OUT}/muril_intent'
model.half().save_pretrained(save_dir); tok.save_pretrained(save_dir)
shutil.copy('labels.json', f'{save_dir}/labels.json')
shutil.make_archive(f'{OUT}/muril_intent', 'zip', save_dir)
print('saved:', save_dir, '| zip:', f'{OUT}/muril_intent.zip', f'({os.path.getsize(OUT + "/muril_intent.zip") / 1e9:.2f} GB)')